In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import plotly.express as px
import plotly.graph_objs as go
import polars as pl
from IPython.display import display  # noqa: A004

### negative log

In [ ]:
x = np.linspace(0, 1, 100)[1:-1]
y = -np.log2(x)
plt.plot(x, y)
plt.plot(x, 1 - x)
plt.show()

# Evaluation Metrics

### nDCG (Normalized Discounted Cumulative Gain)

In [ ]:
def get_ndcg(relevancy: list[int], k: int) -> float:
    dcg: float = 0
    for i, relevance in enumerate(relevancy[:k]):
        dcg += relevance / np.log2(i + 2)
    idcg: float = 0
    for i, relevance in enumerate(sorted(relevancy, reverse=True)[:k]):
        idcg += relevance / np.log2(i + 2)
    return dcg / idcg if idcg > 0 else 0


relevancy: list[int] = [3, 1, 4, 0, 2, 0, 1]
k: int = 5
ndcg = get_ndcg(relevancy, k)
print(f"ndcg: {ndcg:.2f}")


position: np.ndarray = np.arange(10)
discount: np.ndarray = 1 / np.log2(position + 2)
df = pl.DataFrame({"position": position, "discount": discount})
f: go.Figure = px.line(
    df,
    "position",
    "discount",
    markers=True,
    title="nDCG: position discount",
    height=350,
    width=550,
)
f.show()

### MRR (Mean Reciprocal rank)

In [ ]:
def get_mrr(recs: np.ndarray, k: int) -> float:
    top_k_recs = recs[:k]
    return (1 / (np.where(top_k_recs.any(1), top_k_recs.argmax(1), np.inf) + 1)).mean()


k: int = 5
shape: tuple[int, int] = (100, 10)
ones: np.ndarray = np.ones(shape)
mix: np.ndarray = (np.random.rand(*shape) > 0.5).astype(np.int16)
zeros: np.ndarray = np.zeros(shape)

mrr = get_mrr(ones, k=k)
print(f"ones: {mrr:.2f}")
mrr = get_mrr(mix, k=k)
print(f"mix: {mrr:.2f}")
mrr = get_mrr(zeros, k=k)
print(f"zeros: {mrr:.2f}")

### MAP (Mean Average Precision)

In [ ]:
recs = (np.random.rand(3, 5) > 0.5).astype(np.int16)
n_engaged = np.random.randint(0, 10, len(recs)) + recs.sum(1)

non_zero = n_engaged != 0
recs = recs[non_zero]
n_engaged = n_engaged[non_zero]
positions = np.arange(1, recs.shape[1] + 1)[None, :]
map_ = (((recs.cumsum(1) / positions) * recs).sum(1) / n_engaged).mean()

print("recs")
print(recs)
print("n_engaged")
print(n_engaged)
print(f"map: {map_:.2f}")

### Calibration

In [ ]:
def calibration(df: pl.DataFrame, name: str) -> None:
    calibration_ratio = df.select(pl.mean("y_prob") / pl.mean("y_true")).item()
    labels = [str(o) for o in range(10)]
    calibration_curve = (
        df.with_columns(pl.col("y_prob").qcut(10, labels=labels).alias("bin"))
        .group_by("bin")
        .agg(
            pl.len(),
            pl.mean("y_true").alias("pct_true"),
            pl.mean("y_prob").alias("pct_prob"),
        )
        .sort("bin")
    )
    ece = (
        calibration_curve.with_columns(
            ((pl.col("pct_true") - pl.col("pct_prob")).abs() * pl.col("len")).alias(
                "weighted_error"
            )
        )
        .select(pl.sum("weighted_error") / pl.sum("len"))
        .item()
    )
    title: str = f"{name} - ECE: {ece:.2f} - calibration_ratio: {calibration_ratio:.2f}"
    f = px.line(calibration_curve, "pct_prob", "pct_true", title=title, markers=True)
    f.add_shape(
        type="line", x0=0, y0=0, x1=1, y1=1, line={"color": "gray", "dash": "dash"}
    )
    display(f)


n: int = 10_000
df = pl.DataFrame(
    {
        "y_prob": np.random.rand(n),
        "y_true": np.random.randint(0, 2, n),
    }
)
calibration(df, "random")
df = pl.DataFrame(
    {
        "y_prob": np.linspace(0.0001, 0.9999, n),
        "y_true": np.sort(np.random.randint(0, 2, n)),
    }
)
calibration(df, "perfect discrimination")
y_prob = np.random.rand(n)
y_true = np.random.binomial(1, y_prob)
df = pl.DataFrame({"y_prob": y_prob, "y_true": y_true})
calibration(df, "perfect calibration")